# Bounded GPAC Exponentiation: Computing $\alpha^\beta$

**Companion notebook for:** *Bounded Analog Complexity* (DNA 32, 2026)

This notebook simulates the bounded GPAC construction for computing $\alpha^\beta$ (Construction 6.2 in the paper).
We demonstrate two examples:
- $(\pi/4)^e \approx 0.5186$ — where $\alpha < 1$, so intermediate variables go **negative**
- $e^\pi \approx 23.14$ — where $\alpha > 1$, so all intermediates stay **positive**

**Key insight:** The output $z(t) = e^{R(t)}$ is **strictly positive** for all $t \geq 0$, regardless of the sign of the intermediates. This means $z$ does not require dual-rail encoding for CRN implementation — only the intermediate variables that become negative need selective dual-railing ([Haisler et al., UCNC 2025](https://doi.org/...)).

All input constants ($\pi/4$, $e$, $\pi$) are computed from their **actual ODE constructions** from [Huang, Klinge, Lathrop (DNA 25, 2019)](https://doi.org/...).

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'figure.dpi': 150,
})

## 1. The $\alpha^\beta$ Construction (Construction 6.2)

Given upstream GPACs computing $x(t) \to \alpha$ and $y(t) \to \beta$, define:

$$
\begin{aligned}
  x_1' &= (x - 1) - x_1, \\
  u'   &= (1 - v)\,x_1', \\
  v'   &= (1 - v)^2\,x_1', \\
  z'   &= z\,(y'\,u + y\,(1-v)\,x_1'),
\end{aligned}
$$

with $x_1(0) = u(0) = v(0) = 0$ and $z(0) = 1$.

As $t \to \infty$: $x_1 \to \alpha - 1$, $u \to \ln(\alpha)$, $v \to (\alpha-1)/\alpha$, and $z \to e^{\beta \ln \alpha} = \alpha^\beta$.

## 2. DNA25 ODE Constructions for Input Constants

**Computing $e$** via $X + Y \to X + 2Y$ (mass-action ODE):
$$x' = -x, \quad z_e' = x \cdot z_e, \quad x(0) = z_e(0) = 1 \implies z_e(t) \to e$$

**Computing $\pi/4$** via arctan construction:
$$w' = -w, \quad p' = -2wpq, \quad q' = w(p^2 - q^2), \quad z_{\pi/4}' = wp$$
with $w(0) = p(0) = 1$, $q(0) = z_{\pi/4}(0) = 0 \implies z_{\pi/4}(t) \to \pi/4$.

**Computing $\pi$**: same arctan system, but $z_\pi' = 4wp \implies z_\pi(t) \to \pi$.

In [ ]:
def system_pi4_e(t, Y):
    """Full ODE system for (π/4)^e."""
    w_a, x_a, y_a, z_a, x_b, z_b, x1, u, v, Z = Y

    # π/4 via arctan
    dw_a = -w_a
    dx_a = -2 * w_a * x_a * y_a
    dy_a = w_a * x_a**2 - w_a * y_a**2
    dz_a = w_a * x_a

    # e via X+Y→X+2Y
    dx_b = -x_b
    dz_b = x_b * z_b

    # α^β construction
    dx1 = (z_a - 1) - x1
    du = (1 - v) * dx1
    dv = (1 - v)**2 * dx1
    dZ = Z * (dz_b * u + z_b * (1 - v) * dx1)

    return [dw_a, dx_a, dy_a, dz_a, dx_b, dz_b, dx1, du, dv, dZ]


def system_e_pi(t, Y):
    """Full ODE system for e^π."""
    x_a, z_a, w_b, x_b, y_b, z_b, x1, u, v, Z = Y

    # e via X+Y→X+2Y
    dx_a = -x_a
    dz_a = x_a * z_a

    # π via 4*arctan
    dw_b = -w_b
    dx_b = -2 * w_b * x_b * y_b
    dy_b = w_b * x_b**2 - w_b * y_b**2
    dz_b = 4 * w_b * x_b

    # α^β construction
    dx1 = (z_a - 1) - x1
    du = (1 - v) * dx1
    dv = (1 - v)**2 * dx1
    dZ = Z * (dz_b * u + z_b * (1 - v) * dx1)

    return [dx_a, dz_a, dw_b, dx_b, dy_b, dz_b, dx1, du, dv, dZ]

In [ ]:
# Simulate both systems
T = 20
t_eval = np.linspace(0.001, T, 2000)

# (π/4)^e
Y0_1 = [1.0, 1.0, 0.0, 0.0,  # π/4 system: w, x, y, z
         1.0, 1.0,              # e system: x, z
         0.0, 0.0, 0.0, 1.0]   # α^β: x1, u, v, Z
sol1 = solve_ivp(system_pi4_e, [0.001, T], Y0_1,
                 max_step=0.01, rtol=1e-12, atol=1e-14,
                 dense_output=True)
Y1 = sol1.sol(t_eval)

# e^π
Y0_2 = [1.0, 1.0,              # e system: x, z
         1.0, 1.0, 0.0, 0.0,   # π system: w, x, y, z
         0.0, 0.0, 0.0, 1.0]   # α^β: x1, u, v, Z
sol2 = solve_ivp(system_e_pi, [0.001, T], Y0_2,
                 max_step=0.01, rtol=1e-12, atol=1e-14,
                 dense_output=True)
Y2 = sol2.sol(t_eval)

# Targets
target1 = (np.pi / 4) ** np.e
target2 = np.e ** np.pi

print(f'(π/4)^e: computed = {Y1[9][-1]:.8f}, exact = {target1:.8f}, error = {abs(Y1[9][-1]-target1):.2e}')
print(f'e^π:     computed = {Y2[9][-1]:.8f}, exact = {target2:.8f}, error = {abs(Y2[9][-1]-target2):.2e}')

## 3. Visualization: Negative Intermediates, Positive Output

The key point: when $\alpha < 1$ (as in $(\pi/4)^e$), the shifted variable $x_1 \to \alpha - 1 < 0$, the logarithm $u \to \ln \alpha < 0$, and $v \to (\alpha-1)/\alpha < 0$. These negative variables require **dual-rail encoding** for CRN implementation.

But the output $z(t) = e^{R(t)}$ is the exponential of a real number — it is **always positive**. By the **selective dual-railing** technique, only the intermediate variables need dual-rail encoding; the output $z$ is represented directly as a single CRN species concentration.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))

# === Top row: (π/4)^e ===
# (a) Internal variables — negative
ax = axes[0, 0]
ax.plot(t_eval, Y1[6], 'C2', lw=2, label=r'$x_1 \to \pi/4{-}1$')
ax.plot(t_eval, Y1[7], 'C3', lw=2, label=r'$u \to \ln(\pi/4)$')
ax.plot(t_eval, Y1[8], 'C4', lw=2, label=r'$v \to (\alpha{-}1)/\alpha$')
ax.axhline(y=0, color='k', lw=0.8, alpha=0.5)
ax.axhspan(ymin=-0.5, ymax=0, color='red', alpha=0.06)
ax.set_xlabel('Time $t$')
ax.set_title(r'(a) Internal variables $(\pi/4)^e$')
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(0, T); ax.set_ylim(-0.5, 0.15)
ax.annotate('negative region\n(needs dual-rail)',
            xy=(12, -0.15), fontsize=8, color='red', ha='center', style='italic')

# (b) Output z > 0
ax = axes[0, 1]
ax.plot(t_eval, Y1[9], 'C0', lw=2.5,
        label=r'$z(t) \to (\pi/4)^e \approx %.4f$' % target1)
ax.axhline(y=target1, color='gray', ls='--', alpha=0.5)
ax.axhline(y=0, color='k', lw=0.8, alpha=0.5)
ax.fill_between(t_eval, 0, Y1[9], alpha=0.08, color='C0')
ax.set_xlabel('Time $t$'); ax.set_ylabel('$z(t)$')
ax.set_title(r'(b) Output $z(t) > 0$ always')
ax.legend(fontsize=9); ax.set_xlim(0, T); ax.set_ylim(-0.1, 1.2)
ax.annotate('$z = e^{R(t)} > 0$: no dual-rail needed',
            xy=(10, 0.15), fontsize=8, color='C0', ha='center', style='italic')

# (c) Error decay
ax = axes[0, 2]
err1 = np.abs(Y1[9] - target1)
ax.semilogy(t_eval, np.maximum(err1, 1e-15), 'C0', lw=2)
ax.set_xlabel('Time $t$'); ax.set_ylabel(r'$|z(t) - (\pi/4)^e|$')
ax.set_title(r'(c) Error decay $(\pi/4)^e$')
ax.set_xlim(0, T); ax.grid(True, alpha=0.3)

# === Bottom row: e^π ===
# (d) Internal variables — all positive
ax = axes[1, 0]
ax.plot(t_eval, Y2[6], 'C2', lw=2, label=r'$x_1 \to e{-}1$')
ax.plot(t_eval, Y2[7], 'C3', lw=2, label=r'$u \to \ln e = 1$')
ax.plot(t_eval, Y2[8], 'C4', lw=2, label=r'$v \to (e{-}1)/e$')
ax.axhline(y=0, color='k', lw=0.8, alpha=0.5)
ax.set_xlabel('Time $t$')
ax.set_title(r'(d) Internal variables $e^\pi$')
ax.legend(fontsize=8, loc='right'); ax.set_xlim(0, T)
ax.annotate('all positive ($\\alpha > 1$)',
            xy=(12, 0.3), fontsize=8, color='C2', ha='center', style='italic')

# (e) Output z
ax = axes[1, 1]
ax.plot(t_eval, Y2[9], 'C1', lw=2.5,
        label=r'$z(t) \to e^\pi \approx %.2f$' % target2)
ax.axhline(y=target2, color='gray', ls='--', alpha=0.5)
ax.fill_between(t_eval, 0, Y2[9], alpha=0.08, color='C1')
ax.set_xlabel('Time $t$'); ax.set_ylabel('$z(t)$')
ax.set_title(r'(e) Output $z(t) > 0$ always')
ax.legend(fontsize=9); ax.set_xlim(0, T)
ax.annotate('$z = e^{R(t)} > 0$: no dual-rail needed',
            xy=(10, 3), fontsize=8, color='C1', ha='center', style='italic')

# (f) Error decay
ax = axes[1, 2]
err2 = np.abs(Y2[9] - target2)
ax.semilogy(t_eval, np.maximum(err2, 1e-15), 'C1', lw=2)
ax.set_xlabel('Time $t$'); ax.set_ylabel(r'$|z(t) - e^\pi|$')
ax.set_title(r'(f) Error decay $e^\pi$')
ax.set_xlim(0, T); ax.grid(True, alpha=0.3)

fig.suptitle(
    r'Bounded GPAC exponentiation: negative intermediates, strictly positive output',
    fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. The Pass-Through Technique

The exponent $R(t) = y(t) \cdot \ln(1 + x_1(t))$ **passes through** the system: it influences $z$ via its derivative $R'$, but is never tracked as an explicit state variable. This is an instance of a general design technique for bounded GPAC constructions (Remark 6.3 in the paper).

Similarly, in the bounded surrogate compilation (Construction 3.1), the original physical time $t$ and the unbounded variable $f$ both pass through the compiled system without appearing in the final ODE.

## 5. Time Complexity

Both $\pi$ and $e$ are computable in linear time ($\mu(r) = \Theta(r)$) by the DNA25 constructions.
By Theorem A.1, the time modulus of $\alpha^\beta$ is:

$$\mu_{\alpha^\beta}(r) = \max\!\big(\mu_\alpha(r + C),\; \mu_\beta(r + C)\big) + O(1)$$

where $C$ depends only on $\alpha$ and $\beta$. Since both inputs are linear-time, so are $(\pi/4)^e$ and $e^\pi$.

The error plots (c) and (f) above confirm exponential convergence, consistent with $\mu(r) = \Theta(r)$.

---

**Paper:** Xiang Huang. *Bounded Analog Complexity.* DNA 32, 2026.  
**Repository:** [github.com/xiangyazi24/Bounded](https://github.com/xiangyazi24/Bounded)